# Problem Restatement

## The question (Q5)
Past bond programs were structured to recommend projects in each Ward to try to
maximize the likelihood of a successful outcome in a voter election. Can the results
be provided in a map format overlaid on the City Ward Map to make sure we are
getting good representation for each Ward? This may require that we have to look at
more than the top 20 locations in the City. In mapping these
collisions/volumes/collision-volume aggregate, we would envision some sort of
heat map for demonstration purposes.

## How my analysis will answer it
My analysis will enable the City of Norman to identify which wards need the most attention in terms of road safety. Having a ranked list of dangerous and a heat map can possibly reveal patterns about dangerous road sections.

**Method:** every road segment gets a volume-to-capacity (V/C) ratio and a collision count. The segments are then ranked **inside each of the 8 wards** three ways: by V/C, by collisions, and by the City's aggregate (V/C rank + collision rank). Ranking inside each ward, instead of making one citywide top 20, is what guarantees every ward is represented. The maps and heat map are built from these lists in Checkpoint 3.

# Data Loading & Inspection

In [ ]:
# Mount Google Drive and import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

# Load ACOG Count Files
shapelist = []
input_years = ['19','21','23','25','26']
df = {}
pd.set_option("display.max_colwidth", None)
for year in input_years:
  dataframe = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/"+"ACOG FY" + year + ".xlsx")
  df[year] = dataframe
  shapelist.append(df[year])
  print(f"Read dataframe year: {year}")

# Load Collision workbook
crashesdf = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Traffic_Project_OU_FALL_PROJECT.xlsx", sheet_name='Base_Data')
shapelist.append(crashesdf)
print("Read crash list dataframe")

# Load the Code_Book
codebook = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Code_Book.xlsx")
print("Read Code Book")
shapelist.append(codebook)

# Load the GeoJSON
wards = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Council_Wards_4934068591990916100.geojson')
shapelist.append(wards)
print("Read Ward GeoJSON file")

## Data problems I noticed

1. It's curious to me that every ACOG has exactly 100 entries
2. It's unfortunate that the columns names differ from year to year because that means I'll need to rename them.
3. The ACOGs don't record data in the same format. `df['19']` and `df['21']` have `Latest` and `Latest Date` but the others don't.
4. The ACOG vehicles per day has several typos like `3,3339` and `4, 436`
5. The crashes data needs to have the locations translated in order to properly plot them on a heatmap
6. Some crash coordinates are `-361`
7. FY25 and FY26 have 101 rows, not 100. The extra row in each has no ID and no coordinates. One FY26 row has no count at all: it says "Under widening construction".
8. The traffic count is not a number. It is text like `EB 2,913vpd - 37mph / WB 2,651vpd`, with the two directions, the speeds and the date all in one cell.
9. Collisions: 272 rows repeat an `AccidentNumber` that is already listed (the same crash written twice with two directions).
10. Collisions: `Lat`/`Lon` are the coordinates of the nearest intersection, not of the crash, and about 1,700 crashes are on private property (parking lots), not on a road.
11. Code_Book: the collision file stores causes as ID numbers (like `-67028`), so they mean nothing until they are looked up in the Code_Book's `EntryID` column.
12. Ward GeoJSON: no errors found. It has 8 polygons in plain latitude/longitude, the same kind of numbers as the other files. It has more columns than needed, and a road that runs along a ward boundary belongs to two wards.


In [ ]:
# Shows the shapes of each dataframe
i = 0
for item in shapelist:
  print(f"\033[92mItem {i} shape: \033[0m{item.shape}")
  print(f"\033[92mItem {i} columns: \033[0m{item.columns}")
  print(" ")
  i = i + 1


# Data Preparation & Transformation

In [ ]:
# Rename the columns
dictionary = {
    'Loc ID':  'loc_id',
    'LOCAL_ID': 'loc_id',
    'On':      'on_road',
    'ON_ROAD': 'on_road',
    'From': 'from',
    'FROM': 'from',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'Unnamed: 16': 'count_text',
    'Unnamed: 13': 'count_text',
    'Unnamed: 11': 'count_text',
    'LATEST_COUNT': 'count_text',
    'To': 'to',
    'TO': 'to',
    'Functional Class': 'road_type',
    'Latest': 'latest',
    'Latest Date': 'latest_date'
}
for year in df:
    df[year] = df[year].rename(columns=dictionary)

In [ ]:
# Find VPD_max
#### (A.I. was used for this RegEx) ####
DIR = r'(?:NB|SB|EB|WB)'
NUM = r'(\d[\d,]*(?:\s\d{3})?)'
VPD1_RE = DIR + r'\D*?' + NUM
VPD2_RE = r'mph\s*(?:[/-]\s*' + DIR + r'?|' + DIR + r')\s*' + NUM
for frame in df:
  # Fix a typo in FY19 (one digit too many) before reading the numbers (from Tutorial 2)
  df[frame]['count_text'] = df[frame]['count_text'].str.replace('3,3339vpd', '3,339vpd', regex=False)
  df[frame]['VPD1'] = df[frame]['count_text'].str.extract(VPD1_RE)[0]
  df[frame]['VPD2'] = df[frame]['count_text'].str.extract(VPD2_RE)[0]

# Fix Dates
df['26']['road_type'] = '-'
for year in df:
  df[year]['count_text'] = df[year]['count_text'].str.replace(")", "", regex=False)
  df[year]['count_text'] = df[year]['count_text'].str.replace("(", "", regex=False)
for year in df:
  df[year]['date'] = df[year]['count_text'].str.extract(r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\s*$")[0]
  df[year]['date'] = pd.to_datetime(df[year]['date'], errors="coerce")

# create `latest` if necessary
for frame in df:
  if 'latest' not in df[frame].columns:
    df[frame]['latest'] = 0

# Shrink the dataframes
df_reduced = {}
for frame in df:
  df_reduced[frame] = df[frame][['loc_id','date','on_road','from','latitude','longitude','count_text','to','road_type','VPD1','VPD2', 'latest']].copy()
  df_reduced[frame]['year'] = 'FY' + frame        # remember which file each row came from

# Combine the dataframes
big_frame = pd.concat(df_reduced)

In [ ]:
# Find VPD_max
big_frame['VPD1'] = big_frame['VPD1'].str.replace(",", "", regex=False)
big_frame['VPD2'] = big_frame['VPD2'].str.replace(",", "", regex=False)
big_frame['VPD1'] = big_frame['VPD1'].str.replace(" ", "", regex=False)
big_frame['VPD2'] = big_frame['VPD2'].str.replace(" ", "", regex=False)
big_frame['VPD1'] = pd.to_numeric(big_frame['VPD1'], errors='coerce')
big_frame['VPD2'] = pd.to_numeric(big_frame['VPD2'], errors='coerce')
big_frame[['VPD1','VPD2']] = big_frame[['VPD1','VPD2']].fillna(0)
big_frame['VPD_sum'] = big_frame['VPD1'] + big_frame['VPD2']
big_frame['VPD_max'] = big_frame[['latest','VPD_sum']].max(axis=1)
big_frame.drop('VPD1', axis=1, inplace=True)
big_frame.drop('VPD2', axis=1, inplace=True)
big_frame.drop('VPD_sum', axis=1, inplace=True)

In [ ]:
big_frame.head(25)

In [ ]:


from shapely.geometry import Point
import matplotlib.pyplot as plt
wards = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Council_Wards_4934068591990916100.geojson')

fig, ax = plt.subplots(figsize=(10, 8))
wards.plot(ax=ax, column='WARDNO', categorical=True, cmap='Pastel1', edgecolor='black', legend=True)

for i in range(len(wards)):
    shape = wards['geometry'].iloc[i]
    center = shape.representative_point()        # a point that is always inside the shape
    ax.text(center.x, center.y, 'Ward ' + str(wards['WARDNO'].iloc[i]), ha='center', fontsize=11, weight='bold')

ax.set_title('City of Norman — City Council Wards')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()


In [ ]:
def find_ward(lat, lon):
    """Return the ward number that contains (lat, lon), or None."""
    p = Point(lon, lat)                 # careful: Point takes (x = longitude, y = latitude)
    for i in range(len(wards)):
        if wards['geometry'].iloc[i].contains(p):
            return wards['WARDNO'].iloc[i]
    return None

def ward_for_row(row):
    return find_ward(row['latitude'], row['longitude'])

big_frame['ward'] = big_frame.apply(ward_for_row, axis=1)
big_frame.head(10)

## Road segments of at least one mile
*The next three cells are adapted from Tutorial 2 (Steps 6 and 8).*

1. **Clean the road names.** `W Robinson St`, `E Robinson St` and `E Robinson St E` are one road, so each name is reduced to a key like `ROBINSON`. Numbered avenues keep their side of town (`12TH E` and `12TH W` are different roads).
2. **Cut each road into one-mile segments.** Norman's main roads sit on a one-mile grid. A segment is one road between two neighboring grid roads, for example *Robinson: 36th Ave NW – 24th Ave NW*. An east-west road is cut using the count station's longitude, and a north-south road using its latitude.
3. **Keep the study roads and the highest volume.** The City's jargon sheet says to use arterials and section-line roads, not freeways, collectors or local streets. A segment can hold several count stations and several years, so `groupby` + `max` keeps the highest volume, as the City asked.

In [ ]:
# Clean the road names (adapted from Tutorial 2, Step 6)
DATA_FOLDER = "/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/"

REMOVE_WORDS = ['N', 'S', 'E', 'W', 'AVE', 'AVENUE', 'ST', 'STREET', 'RD', 'ROAD', 'BLVD', 'BLV', 'DR', 'PKWY',
                'LN', 'LANE', 'HWY', 'CT', 'CIR', 'PL', 'TRL', 'WAY']
QUADRANTS = ['NE', 'NW', 'SE', 'SW']
NAME_ALIASES = {'CEDAR': 'CEDAR LANE', 'STATE 9': 'SH 9'}

def road_key(name):
    """'W Robinson St' -> 'ROBINSON'     '12th Ave NE' -> '12TH E'"""
    if not isinstance(name, str):          # empty cells are not text
        return ''
    name = name.upper()
    for ch in '.,()':
        name = name.replace(ch, ' ')
    side = ''
    kept = []
    for word in name.split():
        if word in QUADRANTS:
            side = word[1]                 # 'NE' -> 'E'
        elif word not in REMOVE_WORDS:
            kept.append(word)
    key = ' '.join(kept)
    key = NAME_ALIASES.get(key, key)
    if key[:1].isdigit():                  # numbered avenue: add the side of town
        key = key + ' ' + side
    return key

big_frame['road'] = big_frame['on_road'].apply(road_key)

# Three station names in FY23 are garbled, e.g. '(E Lindsey St)12th Ave NW'.
# The station ID never changes, so the other years show the real road.
NAME_FIXES_BY_ID = {'52500-135': 'LINDSEY', '52500-16': '48TH W', '52500-293': 'INDIAN HILLS'}
for station_id, right_name in NAME_FIXES_BY_ID.items():
    big_frame.loc[(big_frame['loc_id'] == station_id) & (big_frame['year'] == 'FY23'), 'road'] = right_name

big_frame[['on_road', 'road']].drop_duplicates().head(12)

In [ ]:
# Cut the roads into one-mile segments (adapted from Tutorial 2, Step 8)
NS_GRID = {   # north-south grid roads: longitude -> name
    -97.5473: '72nd Ave NW/SW', -97.5296: '60th Ave NW/SW', -97.5120: '48th Ave NW/SW',
    -97.4943: '36th Ave NW/SW', -97.4767: '24th Ave NW/SW', -97.4590: '12th Ave NW / Berry Rd',
    -97.4412: 'Porter Ave / Jenkins Ave', -97.4236: '12th Ave NE/SE', -97.4059: '24th Ave NE/SE',
    -97.3882: '36th Ave NE/SE', -97.3706: '48th Ave NE/SE', -97.3530: '60th Ave NE/SE',
    -97.3354: '72nd Ave NE/SE', -97.3180: '84th Ave NE/SE', -97.3004: '96th Ave NE/SE',
    -97.2828: '108th Ave NE/SE', -97.2652: '120th Ave NE/SE', -97.2476: '132nd Ave NE/SE',
    -97.2300: '144th Ave NE/SE', -97.2123: '156th Ave NE/SE', -97.1945: '168th Ave NE/SE', -97.1770: '180th Ave NE/SE'}
EW_GRID = {   # east-west grid roads: latitude -> name
    35.3340: 'County line', 35.3195: 'Stella Rd', 35.3051: 'Bethel Rd', 35.2908: 'Indian Hills Rd',
    35.2763: 'Franklin Rd', 35.2619: 'Tecumseh Rd', 35.2474: 'Rock Creek Rd', 35.2329: 'Robinson St',
    35.2184: 'Main St / Alameda St', 35.2039: 'Lindsey St', 35.1894: 'Imhoff Rd', 35.1751: 'Cedar Lane Rd',
    35.1604: 'Post Oak Rd', 35.1458: 'Etowah Rd', 35.1313: 'Slaughterville Rd'}

NORTH_SOUTH_ROADS = ['BERRY', 'PORTER', 'JENKINS', 'FLOOD', 'CLASSEN', 'CHAUTAUQUA', 'UNIVERSITY',
                     'WEBSTER', 'JAMES GARNER', 'ED NOBLE', 'BROADWAY']

def is_north_south(road):
    return road in NORTH_SOUTH_ROADS or road[:1].isdigit()

def find_between(value, grid):
    """Return the two grid positions on either side of `value`, or (None, None)."""
    positions = sorted(grid.keys())
    for i in range(len(positions) - 1):
        if positions[i] <= value < positions[i + 1]:
            return positions[i], positions[i + 1]
    return None, None

def road_title(road):
    if road[:1].isdigit():
        number, side = road.split()
        return number.lower() + ' Ave ' + ('NE/SE' if side == 'E' else 'NW/SW')
    return road.title()

def segment_name(road, lat, lon):
    """Name of the one-mile segment of `road` that contains the point (lat, lon)."""
    if is_north_south(road):
        grid = EW_GRID                     # a north-south road is cut by the east-west grid roads
        low, high = find_between(lat, grid)
    else:
        grid = NS_GRID
        low, high = find_between(lon, grid)
    if low is None:
        return None
    return road_title(road) + ': ' + grid[low] + ' – ' + grid[high]

def segment_for_count(row):
    return segment_name(row['road'], row['latitude'], row['longitude'])

big_frame['segment'] = big_frame.apply(segment_for_count, axis=1)
big_frame[['year', 'on_road', 'from', 'to', 'segment']].head(10)

In [ ]:
# Keep only the roads in the study, then one row per segment with the highest volume
# (adapted from Tutorial 2, Step 8)
SECTION_LINE_ROADS = ['ROBINSON', 'MAIN', 'ALAMEDA', 'LINDSEY',
                      'IMHOFF', 'CEDAR LANE', 'POST OAK', 'ETOWAH',
                      'TECUMSEH', 'ROCK CREEK', 'FRANKLIN', 'INDIAN HILLS', 'BETHEL', 'STELLA',
                      'BERRY', 'PORTER', 'JENKINS']
FREEWAYS = ['SH 9', 'I 35', 'INTERSTATE']
MANUAL_EXCLUDE = ['DAWS']        # ACOG says Minor Arterial, but it is a ~800 vpd campus street

arterial_roads = big_frame[big_frame['road_type'].isin(['Principal Arterial', 'Minor Arterial'])]['road'].unique()

def in_study(road):
    if road in FREEWAYS or road in MANUAL_EXCLUDE:
        return False
    return road in SECTION_LINE_ROADS or road[:1].isdigit() or road in arterial_roads

big_frame['in_study'] = big_frame['road'].apply(in_study)

print('Count records:', len(big_frame))
print('  no count that year (VPD_max is 0):', (big_frame['VPD_max'] == 0).sum())
print('  no coordinates, so no segment:', big_frame['segment'].isna().sum())
print('  on a road left out of the study:', (~big_frame['in_study']).sum())
print('Roads left out:', sorted(big_frame[~big_frame['in_study']]['road'].unique()))

study = big_frame[big_frame['in_study'] & (big_frame['VPD_max'] > 0) & big_frame['segment'].notna()]

# A segment can have several stations and several years: the highest volume wins
segments = study.groupby('segment').agg(road=('road', 'first'),
                                        volume=('VPD_max', 'max'),
                                        stations=('loc_id', 'nunique'),
                                        latitude=('latitude', 'mean'),
                                        longitude=('longitude', 'mean')).reset_index()
segments['volume'] = segments['volume'].astype(int)
print('Segments:', len(segments))
segments.sort_values('volume', ascending=False).head(10)

## Place each collision on the road where it happened
*The first cell is adapted from Tutorial 1 (Part B).*

* Repeated `AccidentNumber`s are counted once.
* Private-property collisions are left out.
* Collisions with no coordinates (`-361`) get the coordinates of the same intersection from other collisions there (Tutorial 1's "phone book"). The ones that still cannot be placed are left out.
* The coordinates given are the **intersection**, so a "From" collision is moved the stated number of feet north, south, east or west. Distances over a mile (like the `9999` placeholder) are treated as unknown.

Then each collision is given a segment the same way the traffic counts were: its road is `StreetName` (the road the collision was **on**, not the cross street), and its moved location picks the mile. A collision coded at Franklin Rd & 48th Ave NE that happened on 48th Ave belongs to 48th Ave. A collision within about 15 m (49 ft) of a grid intersection counts for **both** segments of its road that meet there, because the intersection is the end of both. Collisions that do not land on a segment with a traffic count go no further, as the City's instructions say.

In [ ]:
# Put each collision at its real location (adapted from Tutorial 1, Part B)
import numpy as np
import math

crashes = crashesdf.copy()
print('Collision records:', len(crashes))

# 1. Some collisions are listed twice with the same AccidentNumber. Keep each one once.
crashes = crashes.drop_duplicates('AccidentNumber')
print('Unique collisions:', len(crashes))

# 2. Private property (parking lots) is not a public road
crashes['private'] = crashes['StreetName'].str.contains('PRIVATE', na=False)

# 3. -361 is the City's code for "no coordinates"
crashes.loc[crashes['Lat'] < -90, 'Lat'] = np.nan
crashes.loc[crashes['Lon'] < -180, 'Lon'] = np.nan
print('  on private property:', crashes['private'].sum())
print('  public road but no coordinates:', (crashes['Lat'].isna() & ~crashes['private']).sum())

# 4. Fill in missing coordinates from other crashes at the same intersection (the "phone book")
TUTORIAL_REMOVE_WORDS = ['N', 'S', 'E', 'W', 'AVE', 'ST', 'RD', 'BLVD', 'DR', 'PKWY', 'LN', 'LANE', 'HWY', 'CT', 'CIR', 'PL', 'TRL', 'WAY']

def clean_name(name):
    """'N FLOOD AVE' -> 'FLOOD'   '24TH AVE NW' -> '24TH NW'"""
    if not isinstance(name, str):
        return ''
    name = name.upper().replace('.', ' ')
    kept_words = []
    for word in name.split():
        if word not in TUTORIAL_REMOVE_WORDS:
            kept_words.append(word)
    return ' '.join(kept_words)

def intersection_key(street1, street2):
    """One name for an intersection, the same no matter which street comes first."""
    names = sorted([clean_name(street1), clean_name(street2)])
    return names[0] + ' & ' + names[1]

def full_street(name, suffix):
    """'12TH' + 'NE' -> '12TH NE'"""
    if isinstance(suffix, str):
        return name + ' ' + suffix
    return name

good_at = crashes[(crashes['From_At'] == 'At') & crashes['Lat'].notna() & crashes['XStreetName'].notna()].copy()
keys = []
for i in range(len(good_at)):
    row = good_at.iloc[i]
    keys.append(intersection_key(full_street(row['StreetName'], row['StreetSuffix']),
                                 full_street(row['XStreetName'], row['XStreetSuffix'])))
good_at['key'] = keys
phone_book = good_at.groupby('key')[['Lat', 'Lon']].median().to_dict('index')

def find_intersection(row):
    """Return (lat, lon) of the crash's intersection from the phone book, or (None, None)."""
    street = row['StreetName']
    cross = row['XStreetName']
    if not isinstance(cross, str):              # no cross street -> try "STREET / CROSS STREET"
        if ' / ' in street:
            parts = street.split(' / ', 1)
            street = parts[0]
            cross = parts[1]
        else:
            return (None, None)
    else:
        street = full_street(street, row['StreetSuffix'])
        cross = full_street(cross, row['XStreetSuffix'])
    found = phone_book.get(intersection_key(street, cross))
    if found is None:
        return (None, None)
    return (found['Lat'], found['Lon'])

to_fix = crashes[crashes['Lat'].isna() & ~crashes['private']]
fixed_count = 0
for index, row in to_fix.iterrows():
    lat, lon = find_intersection(row)
    if lat is not None:
        crashes.loc[index, 'Lat'] = lat
        crashes.loc[index, 'Lon'] = lon
        fixed_count = fixed_count + 1
print('  coordinates filled from the phone book:', fixed_count)

crashes = crashes[~crashes['private'] & crashes['Lat'].notna()].copy()
print('Collisions we can locate:', len(crashes))

# 5. Lat/Lon are the INTERSECTION. Move "From" collisions the stated distance and direction.
FEET_PER_DEGREE_LAT = 364000

def move_point(lat, lon, direction, distance_ft):
    """Move a point `distance_ft` feet in `direction`. Returns (new_lat, new_lon)."""
    if pd.isna(distance_ft) or distance_ft > 5280:        # unknown or placeholder (9999) distance: don't move
        return (lat, lon)
    feet_per_degree_lon = FEET_PER_DEGREE_LAT * math.cos(math.radians(lat))
    if direction == 'DirectionNorth':
        lat = lat + distance_ft / FEET_PER_DEGREE_LAT
    elif direction == 'DirectionSouth':
        lat = lat - distance_ft / FEET_PER_DEGREE_LAT
    elif direction == 'DirectionEast':
        lon = lon + distance_ft / feet_per_degree_lon
    elif direction == 'DirectionWest':
        lon = lon - distance_ft / feet_per_degree_lon
    return (lat, lon)

def crash_location(row):
    if row['From_At'] == 'From':
        return move_point(row['Lat'], row['Lon'], row['Direction'], row['Distance'])
    return (row['Lat'], row['Lon'])

locations = crashes.apply(crash_location, axis=1)
crashes['crash_lat'] = [pair[0] for pair in locations]
crashes['crash_lon'] = [pair[1] for pair in locations]

crashes[['StreetName', 'StreetSuffix', 'XStreetName', 'From_At', 'Direction', 'Distance',
         'Lat', 'Lon', 'crash_lat', 'crash_lon']].head(8)

In [ ]:
# Find the segment(s) each collision belongs to

# The road the collision happened ON is StreetName (+ NE/NW/SE/SW from StreetSuffix).
# Some names hold both streets, e.g. 'N FLOOD AVE / HIGHLAND PKWY': the first one is the road.
# road_key() turns it into the same road names the traffic counts use.
crashes['street'] = crashes['StreetName'].str.split(' / ').str[0] + ' ' + crashes['StreetSuffix'].fillna('')
crashes['road'] = crashes['street'].apply(road_key)

# Only collisions on a road that has traffic counts go further (the City's hint)
on_counted_road = crashes['road'].isin(segments['road'])
print('Collisions on a road with traffic counts:', on_counted_road.sum(), 'of', len(crashes))
counted = crashes[on_counted_road].copy()

# A collision within about 15 m (49 ft) of a grid intersection counts for BOTH segments of the road
# that meet there, because the intersection is the end of both.
NEAR_FT = 49
FEET_PER_DEGREE_LON = FEET_PER_DEGREE_LAT * math.cos(math.radians(35.22))     # Norman's latitude

def segments_for_crash(row):
    """Return a list of the segment names this collision belongs to (usually one, two at a grid intersection)."""
    road = row['road']
    lat = row['crash_lat']
    lon = row['crash_lon']
    found = [segment_name(road, lat, lon)]
    if is_north_south(road):
        for grid_lat in EW_GRID:                      # the east-west grid roads it crosses
            if abs(lat - grid_lat) * FEET_PER_DEGREE_LAT <= NEAR_FT:
                found.append(segment_name(road, grid_lat - 0.000001, lon))   # the segment just south
                found.append(segment_name(road, grid_lat + 0.000001, lon))   # the segment just north
    else:
        for grid_lon in NS_GRID:                      # the north-south grid roads it crosses
            if abs(lon - grid_lon) * FEET_PER_DEGREE_LON <= NEAR_FT:
                found.append(segment_name(road, lat, grid_lon - 0.000001))   # the segment just west
                found.append(segment_name(road, lat, grid_lon + 0.000001))   # the segment just east
    names = []
    for name in found:
        if name is not None and name not in names:
            names.append(name)
    return names

counted['segment'] = counted.apply(segments_for_crash, axis=1)
counted = counted.explode('segment')               # one row per (collision, segment)

# Keep only collisions that land on a mile of road where traffic was actually counted
counted = counted[counted['segment'].isin(segments['segment'])]
print('Collisions on a counted segment:', counted['AccidentNumber'].nunique())
print('  of these, counted for two segments (at a grid intersection):', counted['AccidentNumber'].duplicated().sum())

counted[['StreetName', 'StreetSuffix', 'XStreetName', 'From_At', 'Direction', 'Distance', 'segment']].head(8)

# Metric Computation

## V/C ratio
*Adapted from Tutorial 2 (Steps 9 and 10).*

The number of lanes for each segment comes from `lanes_lookup.csv` (codes like `2`, `4U` = four lanes undivided, `4D` = four lanes divided, `5` = four lanes plus a center turn lane). The lane code is looked up in the ACOG capacity table, and **V/C = volume ÷ capacity**. A V/C of 1.0 means the road carries exactly what it was built for.

The **LOS E** capacity column is used because the jargon sheet defines capacity as the upper limit of traffic before congestion, which is what LOS E measures. It is also the newer (2025) plan.

In [ ]:
# Lanes, capacity and V/C (adapted from Tutorial 2, Steps 9 and 10)
lanes = pd.read_csv(DATA_FOLDER + 'lanes_lookup.csv', dtype=str)

segments = pd.merge(segments, lanes[['segment', 'lanes']], on='segment', how='left', indicator=True)
print(segments['_merge'].value_counts())
print('Segments without lanes:', list(segments[segments['_merge'] == 'left_only']['segment']))

segments['lanes'] = segments['lanes'].fillna('2')       # assume 2 lanes if unknown
segments = segments.drop(columns='_merge')

# ACOG capacity table, LOS E column (2025 OCARTS plan), in vehicles per day
CAPACITY_LOS_E = {'2': 17100, '3': 17100, '4U': 34200, '4D': 38000, '5': 36000,
                  '6U': 52300, '6D': 58000, '1W2': 22000, '1W3': 33000}

segments['capacity'] = segments['lanes'].map(CAPACITY_LOS_E)
segments['vc'] = (segments['volume'] / segments['capacity']).round(3)

segments[['segment', 'volume', 'lanes', 'capacity', 'vc']].sort_values('vc', ascending=False).head(10)

## Collisions per segment

In [ ]:
# Count the collisions on each segment
collision_counts = counted.groupby('segment').size().reset_index(name='collisions')

segments = pd.merge(segments, collision_counts, on='segment', how='left')
segments['collisions'] = segments['collisions'].fillna(0).astype(int)     # no match = 0 collisions

print('Segments with at least one collision:', (segments['collisions'] > 0).sum(), 'of', len(segments))
segments[['segment', 'volume', 'vc', 'collisions']].sort_values('collisions', ascending=False).head(10)

## Wards
Each segment is drawn as a line between its two grid roads. A segment belongs to **every ward that holds at least 10% of it**, so a road that runs **along** a ward boundary is counted in **both** wards, as the City asked. Each ward's area is widened by about 70 feet first, because the grid coordinates and the ward map's edges do not line up exactly; without that, a road on a boundary would land in only one of its two wards.

In [ ]:
# Which ward(s) is each segment in?
from shapely.geometry import LineString

MIN_SHARE = 0.10      # a ward must hold at least 10% of the segment
EDGE = 0.0002         # about 70 feet: a road on a ward boundary sits right on the edge of both wards

def segment_line(row):
    """The segment as a straight line between its two grid roads."""
    if is_north_south(row['road']):
        low, high = find_between(row['latitude'], EW_GRID)
        return LineString([(row['longitude'], low), (row['longitude'], high)])
    low, high = find_between(row['longitude'], NS_GRID)
    return LineString([(low, row['latitude']), (high, row['latitude'])])

def segment_wards(row):
    """Return a list of every ward that holds at least 10% of the segment."""
    line = segment_line(row)
    found = []
    for i in range(len(wards)):
        ward_area = wards['geometry'].iloc[i].buffer(EDGE)
        share = line.intersection(ward_area).length / line.length
        if share >= MIN_SHARE:
            found.append(int(wards['WARDNO'].iloc[i]))
    return found

segments['wards'] = segments.apply(segment_wards, axis=1)
segments['number_of_wards'] = segments['wards'].apply(len)
print(segments['number_of_wards'].value_counts().sort_index())

# One row per (segment, ward): a segment in two wards appears in both wards' lists
by_ward = segments.explode('wards').rename(columns={'wards': 'ward'})
by_ward = by_ward[by_ward['ward'].notna()].copy()
by_ward['ward'] = by_ward['ward'].astype(int)
print('Rows in the ward table:', len(by_ward))
by_ward.groupby('ward').size()

## Rankings
Three rankings, each made separately inside every ward (`groupby('ward')`), so there are 8 lists of each kind:

1. **V/C rank**: highest V/C is 1
2. **Collision rank**: most collisions is 1
3. **Aggregate**: for segments that have both a volume and collisions, add the two rank numbers. The smallest sum is the top priority. (City's example: 2nd in collisions + 4th in V/C = 6.)

Ties share the best rank (`method='min'`: 1, 2, 2, 4). In the aggregate lists, the V/C and collision ranks are recomputed among the segments that have both, and a tie in the sum goes to the segment with the higher V/C.

In [ ]:
# The three rankings, separately inside each ward

# 1. V/C: highest V/C = rank 1 (ties share the best rank: 1, 2, 2, 4)
by_ward['vc_rank'] = by_ward.groupby('ward')['vc'].rank(ascending=False, method='min').astype(int)

# 2. Collisions: most collisions = rank 1
by_ward['collision_rank'] = by_ward.groupby('ward')['collisions'].rank(ascending=False, method='min').astype(int)

# 3. Aggregate: only segments with BOTH volume and at least 1 collision, ranked again within that set.
#    Add the two rank numbers; the smallest sum is the top priority. Ties go to the higher V/C.
both = by_ward[by_ward['collisions'] > 0].copy()
both['vc_rank_both'] = both.groupby('ward')['vc'].rank(ascending=False, method='min')
both['collision_rank_both'] = both.groupby('ward')['collisions'].rank(ascending=False, method='min')
both['rank_sum'] = (both['vc_rank_both'] + both['collision_rank_both']).astype(int)
both = both.sort_values(['ward', 'rank_sum', 'vc'], ascending=[True, True, False])
both['priority_rank'] = both.groupby('ward').cumcount() + 1

by_ward = pd.merge(by_ward, both[['segment', 'ward', 'rank_sum', 'priority_rank']], on=['segment', 'ward'], how='left')

# The same aggregate for the whole city, to compare with the ward lists later
city = segments[segments['collisions'] > 0].copy()
city['city_rank_sum'] = (city['vc'].rank(ascending=False, method='min')
                         + city['collisions'].rank(ascending=False, method='min')).astype(int)
city = city.sort_values(['city_rank_sum', 'vc'], ascending=[True, False])
city['city_rank'] = range(1, len(city) + 1)

print('Rows ranked in the aggregate lists:', by_ward['priority_rank'].notna().sum(), 'of', len(by_ward))
by_ward.sort_values(['ward', 'priority_rank']).head(10)

# Business Question

**Question (Q5):** can the results be shown by ward, so that every ward is well represented, even if that means looking beyond the top 20 locations in the city?

**Answer: yes, and it is necessary.** The lists below rank the segments inside each ward, so every ward has its own number 1, 2, 3 and so on. The table after the lists shows why a single citywide top 20 would not work.

In [ ]:
# The ranked lists for each ward (top 5 shown; the full lists are saved below)
for w in range(1, 9):
    this_ward = by_ward[by_ward['ward'] == w]
    print()
    print('=============== WARD', w, '(', len(this_ward), 'segments ) ===============')
    print('--- Aggregate priority (V/C rank + collision rank) ---')
    print(this_ward.sort_values('priority_rank')[['priority_rank', 'segment', 'vc', 'collisions', 'rank_sum']].head(5).to_string(index=False))
    print('--- By V/C ---')
    print(this_ward.sort_values('vc_rank')[['vc_rank', 'segment', 'volume', 'lanes', 'vc']].head(5).to_string(index=False))
    print('--- By collisions ---')
    print(this_ward.sort_values('collision_rank')[['collision_rank', 'segment', 'collisions']].head(5).to_string(index=False))

### Why one citywide top 20 is not enough
The table below counts how many of the citywide top 20 segments (by the same aggregate) fall in each ward.

In [ ]:
# Would a single citywide top 20 represent every ward?
top20 = city.sort_values('city_rank').head(20)
top20_by_ward = top20.explode('wards').groupby('wards').size()

representation = pd.DataFrame({'segments in the ward': by_ward.groupby('ward').size(),
                               'in the citywide top 20': top20_by_ward}).fillna(0).astype(int)
representation.index.name = 'ward'
representation

**What the results show**

* **Ward 5 has no segment in the citywide top 20**, even though it has more counted segments than any other ward (33). It is the large rural east side: low volumes and few collisions. A citywide list would give Ward 5 voters nothing. Its own list still has clear leaders: *Classen: Post Oak Rd – Cedar Lane Rd* and *24th Ave SE: Imhoff Rd – Lindsey St*.
* The citywide top 20 is concentrated in the center of town: Ward 2 has 7 of the 20 and Wards 4 and 8 have 6 each (a segment on a boundary counts for every ward it is in).
* The same few roads are at or near the top of several wards' lists, mostly because they run along ward boundaries: **Robinson St**, **12th Ave NE/SE**, **Classen Blvd** and **Main St**. For example, *Robinson: 36th Ave NW – 24th Ave NW* is number 1 in Wards 2 and 3 and number 2 in Ward 8. One project there would serve three wards. Citywide, the top segment is *12th Ave NE/SE: Main St – Robinson St* (3rd in V/C + 2nd in collisions = 5), which leads both Ward 4 and Ward 6.
* The highest V/C in the city is *12th Ave NE: Rock Creek Rd – Tecumseh Rd* (0.97, a two-lane road), but it has fewer collisions, so it is number 2 in Ward 6 instead of number 1. No segment is over capacity (V/C above 1.0).

These choices follow the course's *Norman Traffic Case Domain Summary*, and the results reproduce its examples: 680 collisions on *Main: 36th Ave NW – 24th Ave NW*, and *12th Ave NE/SE: Main St – Robinson St* as the citywide number 1 with a score of 5.

**Key choices and why**

| Choice | Why |
|---|---|
| Volume = the higher of `Latest` and the two directions added together, then the highest across stations and years in a segment | The City's rule: use the highest volume available |
| One-mile segments from the road grid | The City wants segments of at least a mile; the grid gives the same segment for traffic counts and collisions |
| Arterials and section-line roads only | The jargon sheet excludes freeways, collectors and local streets |
| LOS E capacity | It matches the definition of capacity (the limit before congestion) and is the newer plan |
| Missing collision coordinates filled from other collisions at the same intersection | Keeps as many collisions as possible (Tutorial 1) |
| Collisions assigned by `StreetName` and the moved location | The collision belongs to the road it was on, not the cross street |
| A collision within 15 m of a grid intersection counts for both segments that meet there | The intersection is the end of both segments |
| A segment belongs to every ward holding at least 10% of it | The City's instruction to count a boundary segment in both wards |
| Rank-sum aggregate inside each ward, ties going to the higher V/C | The City's method, applied per ward to answer Q5 |

**Limits of this analysis**

* The lane counts in `lanes_lookup.csv` are estimates that still need to be checked on the City's aerial map. Lanes change V/C a lot.
* Of the 611 collisions on public roads with no coordinates, 178 are recovered from the phone book. The other 433 (mostly I-35 ramps, which are not in the study) are left out.
* A collision at a grid intersection counts for both segments that meet there, so the per-segment counts add up to more than the number of collisions.
* Collisions cover 2016 to August 2026 and are simple counts, not adjusted for how much traffic a road carries.

In [ ]:
# Save the full lists: one Excel sheet per ward, and one table for the maps in Checkpoint 3
columns = ['ward', 'segment', 'volume', 'lanes', 'capacity', 'vc', 'vc_rank',
           'collisions', 'collision_rank', 'rank_sum', 'priority_rank']

with pd.ExcelWriter(DATA_FOLDER + 'Group101_ward_rankings.xlsx') as writer:
    for w in range(1, 9):
        this_ward = by_ward[by_ward['ward'] == w].sort_values(['priority_rank', 'vc_rank'])
        this_ward[columns].to_excel(writer, sheet_name='Ward ' + str(w), index=False)

by_ward.to_csv(DATA_FOLDER + 'Group101_segments_by_ward.csv', index=False)
print('Saved Group101_ward_rankings.xlsx and Group101_segments_by_ward.csv')

# Visualizations (Checkpoint 3)
Four views that answer Q5, then all four on one dashboard. Every map uses the City Council ward map underneath, at the full city width so the large Ward 5 on the east side is visible.

| # | View | What it shows |
|---|---|---|
| 1 | Collision heat map | Where collisions happen (the heat map the City asked for) |
| 2 | Congestion map | Each one-mile segment colored by its V/C ratio |
| 3 | Ward priority map | Each ward's own #1, #2 and #3 by combined priority |
| 4 | Ward representation chart | Why a single citywide top 20 would leave a ward out |

Each chart is also saved as a PNG in the Drive folder for the presentation slides.

*Colors: orange for collisions and blue for V/C, each light (low) to dark (high); red highlights priorities, with gray for everything else.*

In [ ]:
# Shared map settings and a function that draws the ward map underneath every chart
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.lines import Line2D

SURFACE = '#fcfcfb'      # chart background
INK = '#0b0b0b'          # main text
INK_2 = '#52514e'        # secondary text
WARD_FILL = '#f0efec'
WARD_EDGE = '#9a9893'

# One hue per job, light = low and dark = high
VC_RAMP = LinearSegmentedColormap.from_list('vc', ['#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b'])
CRASH_RAMP = LinearSegmentedColormap.from_list('crash', ['#fde6d8', '#f8b893', '#f08a57', '#eb6834', '#c04a17', '#86300c'])
TOP1, TOP3, OTHER = '#b3262b', '#ee8b88', '#c9c7c1'     # priority map: #1, #2-3, everything else

minx, miny, maxx, maxy = wards.total_bounds
MAP_XLIM = (minx - 0.005, maxx + 0.005)
MAP_YLIM = (miny - 0.005, maxy + 0.005)

LABEL_NUDGE = {1: (0.008, 0.0)}     # move the Ward 1 label off Classen Blvd

def draw_wards(ax):
    """Gray ward map with ward numbers, and the same frame on every map."""
    wards.plot(ax=ax, color=WARD_FILL, edgecolor=WARD_EDGE, linewidth=1)
    for i in range(len(wards)):
        center = wards['geometry'].iloc[i].representative_point()
        ward = int(wards['WARDNO'].iloc[i])
        dx, dy = LABEL_NUDGE.get(ward, (0, 0))
        ax.text(center.x + dx, center.y + dy, 'Ward ' + str(ward), ha='center', va='center',
                fontsize=10, color=INK_2, weight='bold', zorder=5,
                bbox={'facecolor': SURFACE, 'alpha': 0.75, 'edgecolor': 'none', 'pad': 1.5})
    ax.set_xlim(MAP_XLIM)
    ax.set_ylim(MAP_YLIM)
    ax.set_aspect(1 / math.cos(math.radians(35.24)))   # so a mile east looks as long as a mile north
    ax.set_axis_off()

def segment_lines(table):
    """Turn each segment into a line ((x1, y1), (x2, y2)) for drawing."""
    lines = []
    for _, row in table.iterrows():
        line = segment_line(row)
        lines.append(list(line.coords))
    return lines

def style_title(ax, title, subtitle):
    ax.set_title(title + '\n', fontsize=14, color=INK, weight='bold', loc='left')
    ax.text(0, 1.0, subtitle, transform=ax.transAxes, fontsize=10, color=INK_2, va='bottom')

plt.rcParams['figure.facecolor'] = SURFACE
plt.rcParams['axes.facecolor'] = SURFACE
plt.rcParams['font.size'] = 10

# The best (smallest) priority rank each segment has in any of its wards
best_rank = by_ward.groupby('segment')['priority_rank'].min()
segments['best_priority_rank'] = segments['segment'].map(best_rank)
print('Ready to draw:', len(segments), 'segments,', len(crashes), 'located collisions')

In [ ]:
# Visualization 1: where collisions happen (heat map)
def draw_collision_heatmap(ax, fig):
    draw_wards(ax)
    cells = ax.hexbin(crashes['crash_lon'], crashes['crash_lat'], gridsize=90, extent=MAP_XLIM + MAP_YLIM,
                      cmap=CRASH_RAMP, norm=LogNorm(), mincnt=1, linewidths=0.2, edgecolors=SURFACE, zorder=3)
    bar = fig.colorbar(cells, ax=ax, shrink=0.6, pad=0.01)
    bar.set_label('Collisions in each hexagon (log scale)', color=INK_2)
    bar.outline.set_visible(False)
    style_title(ax, 'Collisions cluster in central and west Norman',
                format(len(crashes), ',') + ' located collisions on public roads, Jan 2016 – Aug 2026')

fig, ax = plt.subplots(figsize=(13, 8))
draw_collision_heatmap(ax, fig)
plt.savefig(DATA_FOLDER + 'viz1_collision_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization 2: how full each road is (V/C ratio)
def draw_vc_map(ax, fig):
    draw_wards(ax)
    ordered = segments.sort_values('vc')                      # draw the fullest roads on top
    roads = LineCollection(segment_lines(ordered), array=ordered['vc'].values, cmap=VC_RAMP,
                           norm=plt.Normalize(0, 1), linewidths=3.5, capstyle='round', zorder=4)
    ax.add_collection(roads)
    bar = fig.colorbar(roads, ax=ax, shrink=0.6, pad=0.01)
    bar.set_label('V/C ratio (1.0 = at capacity)', color=INK_2)
    bar.outline.set_visible(False)
    fullest = segments.sort_values('vc', ascending=False).iloc[0]
    style_title(ax, 'The busiest roads run through the center of the city',
                str(len(segments)) + ' one-mile segments; fullest: ' + fullest['segment'] + ' (V/C ' + str(fullest['vc']) + ')')

fig, ax = plt.subplots(figsize=(13, 8))
draw_vc_map(ax, fig)
plt.savefig(DATA_FOLDER + 'viz2_vc_map.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization 3: each ward's own top priorities (V/C rank + collision rank, ranked inside the ward)

# The #1 segments, each with the ward(s) it leads, numbered for the map
number_ones = by_ward[by_ward['priority_rank'] == 1].groupby('segment')['ward'].apply(list).reset_index()
number_ones = number_ones.merge(segments, on='segment')
number_ones['city_rank'] = number_ones['segment'].map(city.set_index('segment')['city_rank'])
number_ones = number_ones.sort_values('city_rank').reset_index(drop=True)
number_ones['badge'] = range(1, len(number_ones) + 1)

def draw_priority_map(ax, fig):
    draw_wards(ax)
    ax.set_ylim(MAP_YLIM[0], MAP_YLIM[1] + 0.055)       # extra room at the top for the key
    other = segments[segments['best_priority_rank'] > 3]
    top3 = segments[(segments['best_priority_rank'] >= 2) & (segments['best_priority_rank'] <= 3)]
    top1 = segments[segments['best_priority_rank'] == 1]
    ax.add_collection(LineCollection(segment_lines(other), colors=OTHER, linewidths=2, capstyle='round', zorder=3))
    ax.add_collection(LineCollection(segment_lines(top3), colors=TOP3, linewidths=4.5, capstyle='round', zorder=4))
    ax.add_collection(LineCollection(segment_lines(top1), colors=TOP1, linewidths=6, capstyle='round', zorder=6))

    # A numbered marker in the middle of each #1 segment, and a key that names it
    key = []
    for _, row in number_ones.iterrows():
        middle = segment_line(row).interpolate(0.5, normalized=True)
        ax.scatter(middle.x, middle.y, s=260, color=TOP1, edgecolor=SURFACE, linewidth=2, zorder=7)
        ax.text(middle.x, middle.y, str(row['badge']), ha='center', va='center', fontsize=9,
                color='white', weight='bold', zorder=8)
        ward_text = ', '.join(str(w) for w in row['ward'])
        key.append(str(row['badge']) + '   ' + row['segment'] + '   (Ward' + ('s ' if len(row['ward']) > 1 else ' ') + ward_text + ')')
    ax.text(0.01, 0.99, '#1 in its ward\n' + '\n'.join(key), transform=ax.transAxes, va='top', ha='left',
            fontsize=9.5, color=INK, linespacing=1.6, zorder=9,
            bbox={'facecolor': SURFACE, 'alpha': 0.9, 'edgecolor': 'none', 'pad': 4})

    legend = [Line2D([0], [0], color=TOP1, linewidth=6, label='#1 in its ward'),
              Line2D([0], [0], color=TOP3, linewidth=4.5, label='#2 or #3 in its ward'),
              Line2D([0], [0], color=OTHER, linewidth=2, label='Other ranked segments')]
    ax.legend(handles=legend, loc='lower right', frameon=False, fontsize=10, labelcolor=INK)
    style_title(ax, 'Every ward has its own top projects',
                'Combined priority (V/C rank + collision rank), ranked inside each ward; boundary roads count for both wards')

fig, ax = plt.subplots(figsize=(13, 8))
draw_priority_map(ax, fig)
plt.savefig(DATA_FOLDER + 'viz3_ward_priority_map.png', dpi=200, bbox_inches='tight')
plt.show()

number_ones[['badge', 'segment', 'ward', 'vc', 'collisions', 'city_rank']]

In [ ]:
# Visualization 4: would a citywide top 20 represent every ward?
def draw_representation(ax, fig):
    table = representation.sort_index(ascending=False)          # Ward 1 at the top
    labels = ['Ward ' + str(w) + '  (' + str(n) + ' segments)' for w, n in
              zip(table.index, table['segments in the ward'])]
    values = table['in the citywide top 20']
    colors = [TOP1 if v == 0 else '#3987e5' for v in values]
    bars = ax.barh(labels, values, color=colors, height=0.6)
    for bar, v in zip(bars, values):
        text = str(v) if v > 0 else '0  — no segment in the citywide top 20'
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2, text, va='center',
                fontsize=11, color=INK, weight='bold' if v == 0 else 'normal')
    ax.set_xlim(0, values.max() + 4)
    ax.set_xlabel('Segments in the citywide top 20 (a boundary segment counts for each of its wards)', color=INK_2)
    ax.tick_params(colors=INK_2, length=0, labelsize=11)
    for side in ['top', 'right', 'left']:
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color(WARD_EDGE)
    ax.grid(axis='x', color='#e4e2dd', linewidth=0.8)
    ax.set_axisbelow(True)
    style_title(ax, 'A single citywide top 20 would skip Ward 5',
                'Ward 5 has the most counted segments but none rank in the citywide top 20 — ranking inside each ward fixes this')

fig, ax = plt.subplots(figsize=(13, 6))
draw_representation(ax, fig)
plt.savefig(DATA_FOLDER + 'viz4_ward_representation.png', dpi=200, bbox_inches='tight')
plt.show()

### Dashboard: all four on one page

In [ ]:
# Dashboard: the four visualizations on one page
fig, axes = plt.subplots(2, 2, figsize=(26, 17), gridspec_kw={'hspace': 0.18, 'wspace': 0.3})
draw_collision_heatmap(axes[0, 0], fig)
draw_vc_map(axes[0, 1], fig)
draw_priority_map(axes[1, 0], fig)
draw_representation(axes[1, 1], fig)
fig.suptitle('City of Norman: road congestion and collisions by City Council ward\n', fontsize=22, weight='bold',
             color=INK, x=0.125, ha='left', y=0.975)
fig.text(0.125, 0.935, format(len(crashes), ',') + ' collisions (2016 – Aug 2026)   |   ' + str(len(segments))
         + ' one-mile road segments   |   8 wards   |   Citywide #1: ' + city.iloc[0]['segment'],
         fontsize=13, color=INK_2)
plt.savefig(DATA_FOLDER + 'dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# AI Tutor Use
I used Claude (an AI assistant) throughout this checkpoint, and for more than debugging.

**What the AI wrote**
* The regular expressions that pull the two directional counts and the date out of the count text.
* All of the code from "Road segments of at least one mile" to the end: road names, segments, collision locations, matching collisions to segments, lanes and V/C, wards, rankings, and saving the files. Most of it is adapted from Tutorials 1 and 2, and those cells say so.
* The written explanations in those sections, the added inspection notes (7–12), and the Business Question write-up.
* The four visualizations and the dashboard (Checkpoint 3), and their descriptions.

**What I wrote, with debugging help**
* Loading the files, the inspection loop, the rename dictionary, combining the five years, and `VPD_max`.

**Questions I asked and what I changed**
1. *Why were my VPD columns all `None`?* I had used `fillna(0, inplace=True)` and also assigned the result back. `inplace=True` returns `None`, so I removed it.
2. *How do I keep the higher of `Latest` and the summed directions?* An `if` cannot compare two whole columns, so I changed it to `max(axis=1)`.
3. *Why did `find_ward` fail on my table?* It takes one point, and I was passing whole columns. I changed it to run once per row with `.apply(..., axis=1)`.